# 🍷 Análise Comparativa de Outliers — Wine Dataset
Este notebook aplica **vários detectores de outliers** do PyOD ao *Wine Dataset* (UCI) e compara os resultados usando gráficos, PCA e métricas de avaliação.

In [2]:
# 🛠 Instale as dependências se ainda não estiverem instaladas
# !pip install pyod scikit-learn pandas matplotlib seaborn

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc

from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM
from pyod.models.ecod import ECOD
from pyod.models.hbos import HBOS
from pyod.models.abod import ABOD
#from pyod.models.fastabod import FastABOD
from pyod.models.copod import COPOD
from pyod.models.mad import MAD
from pyod.models.sos import SOS

## 1) Carregar o Wine Dataset

In [11]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data"
columns = [
    "Class label", "Alcohol", "Malic acid", "Ash", "Alcalinity of ash",
    "Magnesium", "Total phenols", "Flavanoids", "Nonflavanoid phenols",
    "Proanthocyanins", "Color intensity", "Hue", "OD280/OD315", "Proline"
]
df = pd.read_csv(url, header=None, names=columns)
df.head()

,Class label,Alcohol,Malic acid,Ash,Alcalinity of ash,Magnesium,Total phenols,Flavanoids,Nonflavanoid phenols,Proanthocyanins,Color intensity,Hue,OD280/OD315,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


## 2) Preparar dados

In [12]:
X = df.drop("Class label", axis=1).values
print(f"Shape dos dados: {X.shape}")

Shape dos dados: (178, 13)


## 3) Definir e treinar detectores de outliers

In [ ]:
detectors = {
    "KNN": KNN(contamination=contamination),
    "LOF": LOF(contamination=contamination),
    "IsolationForest": IForest(contamination=contamination),
    "OneClassSVM": OCSVM(contamination=contamination),
    "ECOD": ECOD(contamination=contamination),
    "HBOS": HBOS(contamination=contamination),
    "ABOD (fast)": ABOD(contamination=contamination, method='fast'),
    "COPOD": COPOD(contamination=contamination),
    "SOS": SOS(contamination=contamination)
}

results = {}
for name, clf in detectors.items():
    clf.fit(X)
    labels = clf.labels_
    scores = clf.decision_scores_
    results[name] = {'labels': labels, 'scores': scores}
    print(f"{name}: {labels.sum()} outliers ({labels.sum()/len(labels)*100:.1f}%)")


KNN: 9 outliers (5.1%)
LOF: 9 outliers (5.1%)
IsolationForest: 9 outliers (5.1%)
OneClassSVM: 9 outliers (5.1%)
ECOD: 9 outliers (5.1%)
HBOS: 9 outliers (5.1%)
ABOD (fast): 9 outliers (5.1%)
COPOD: 9 outliers (5.1%)


ValueError: MAD algorithm is just for univariate data. Got Data with 13 Dimensions.

## 4) Visualização com PCA (2 componentes)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(14, 10))
for i, (name, vals) in enumerate(results.items(), 1):
    plt.subplot(3, 4, i)
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=vals['labels'], cmap='coolwarm', edgecolor='k')
    plt.title(name)
    plt.xlabel('PCA 1')
    plt.ylabel('PCA 2')
plt.tight_layout()
plt.show()

## 5) Scores de Anomalia Ordenados

In [ ]:
plt.figure(figsize=(10, 6))
for name, vals in results.items():
    scores = np.sort(vals['scores'])
    plt.plot(scores, label=name)
plt.title("Scores de Anomalia Ordenados")
plt.xlabel("Índice")
plt.ylabel("Score")
plt.legend()
plt.show()

## 6) Curvas ROC / AUC

In [ ]:
plt.figure(figsize=(10, 6))
for name, vals in results.items():
    fpr, tpr, _ = roc_curve(vals['labels'], vals['scores'])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.2f})")
plt.plot([0,1], [0,1], '--', color='gray')
plt.title("Curvas ROC por Modelo")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## 📌 Conclusão
Este notebook compara **vários métodos de detecção de outliers** aplicados ao mesmo conjunto de dados, com suporte visual e métricas. Você pode adaptar os parâmetros e estender para outros datasets facilmente!